In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
fact_telemetry = spark.read.table(f"{GOLD}.fact_telemetry_30min")

# KPI: FUEL ANOMALY (Rolling Z-Score on Time Series)
w_rolling = Window.partitionBy("machine_sk").orderBy("time_bucket_30m").rowsBetween(-48, 0) # Last 24 hours
fuel_anomaly = fact_telemetry.select("machine_sk", "time_bucket_30m", "avg_fuel") \
    .withColumn("roll_avg", avg("avg_fuel").over(w_rolling)) \
    .withColumn("roll_std", stddev("avg_fuel").over(w_rolling)) \
    .withColumn("z_score", when(col("roll_std") > 0, (col("avg_fuel") - col("roll_avg")) / col("roll_std")).otherwise(0)) \
    .withColumn("is_anomaly", when(abs(col("z_score")) > 2.5, True).otherwise(False))
fuel_anomaly.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.kpi_fuel_anomaly")

# KPI: DAILY COST PER HOUR
maint = spark.read.table(f"{GOLD}.fact_maintenance_event").groupBy("machine_sk", "date_sk").agg(sum("maint_cost").alias("daily_cost"))
util = spark.read.table(f"{GOLD}.fact_utilization_daily").select("machine_sk", "date_sk", "active_hours")
cost_per_hr = util.join(maint, ["machine_sk", "date_sk"], "left").fillna(0, subset=["daily_cost"]) \
    .withColumn("cost_per_hour", when(col("active_hours") > 0, col("daily_cost") / col("active_hours")).otherwise(0))
cost_per_hr.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.kpi_cost_per_hour")
print("✅ gold_07_kpi_efficiency built.")